<a href="https://colab.research.google.com/github/zero-zero99/-/blob/main/PlaywithMetric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.32.0 which is incompatible.


In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

In [4]:
import keras
from keras import layers

In [5]:
from keras.datasets import mnist

def get_mnist_model():
    inputs = keras.Input(shape=(28 * 28,))
    features = layers.Dense(512, activation="relu")(inputs)
    features = layers.Dropout(0.5)(features)
    outputs = layers.Dense(10, activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    return model

(images, labels), (test_images, test_labels) = mnist.load_data()
images = images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255
train_images, val_images = images[10000:], images[:10000]
train_labels, val_labels = labels[10000:], labels[:10000]

model = get_mnist_model()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images,
    train_labels,
    epochs=3,
    validation_data=(val_images, val_labels),
)
test_metrics = model.evaluate(test_images, test_labels)
predictions = model.predict(test_images)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9110 - loss: 0.2918 - val_accuracy: 0.9588 - val_loss: 0.1351
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9558 - loss: 0.1474 - val_accuracy: 0.9696 - val_loss: 0.1024
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9644 - loss: 0.1148 - val_accuracy: 0.9745 - val_loss: 0.0892
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9738 - loss: 0.0847
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


Try to understand "sparse_categorical_crossentropy" and "accuracy"

In [6]:
import keras
import numpy as np

# True classes: 0 for the first sample, 1 for the second, and 2 for the third. // 정답 클래스: 첫 번째는 0, 두 번째는 1, 세 번째는 2
targets = np.array([0, 1, 2])

# Predicted probabilities for each class. // 각 클래스에 대한 예측 확률
example_predictions = np.array([
    [0.8, 0.1, 0.1],  # Predicted class 0 -> Correct
    [0.1, 0.7, 0.2],  # Predicted class 1 -> Correct
    [0.2, 0.6, 0.2],  # Predicted class 1 -> Incorrect
])

In [7]:
ce_metric = keras.metrics.SparseCategoricalCrossentropy(
    from_logits=False
)
ce_metric.update_state(targets, example_predictions)

print(f"Crossentropy: {float(ce_metric.result()):.4f}")

# Calculate manually
correct_probs = example_predictions[np.arange(len(targets)), targets]
individual_ce = -np.log(correct_probs)

print("Probability for correct label (정답 클래스의 예측 확률):", correct_probs)
print("Crossentropy for each item:", np.round(individual_ce, 4))
print(f"Mean: {individual_ce.mean():.4f}")

Crossentropy: 0.7298
Probability for correct label (정답 클래스의 예측 확률): [0.8 0.7 0.2]
Crossentropy for each item: [0.2231 0.3567 1.6094]
Mean: 0.7298


In [ ]:
acc_metric = keras.metrics.SparseCategoricalAccuracy()
acc_metric.update_state(targets, example_predictions)

print(f"Accuracy: {float(acc_metric.result()):.4f}")

# Calculate manually
predicted_classes = np.argmax(example_predictions, axis=1)
is_correct = predicted_classes == targets

print("Predicted classes:", predicted_classes)
print("Correctness:", is_correct)
print(f"Accuracy: {is_correct.mean():.4f}")

Accuracy: 0.6667
Predicted classes: [0 1 1]
Correctness: [ True  True False]
Accuracy: 0.6667


Other metrics from https://keras.io/api/metrics/

In [8]:
metric = keras.metrics.SparseCategoricalAccuracy()
targets = [0, 1, 2]
predictions = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
metric.update_state(targets, predictions)
current_result = metric.result()
print(f"result: {current_result:.2f}")

result: 1.00


In [10]:
targets = [0, 1, 1]
predictions = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
metric.update_state(targets, predictions)
current_result = metric.result()
print(f"result: {current_result:.2f}")

result: 0.83


In [11]:
values = [0, 1, 2, 3, 4]
mean_tracker = keras.metrics.Mean()
for value in values:
    mean_tracker.update_state(value)
print(f"Mean of values: {mean_tracker.result():.2f}")

Mean of values: 2.00
